In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from langdetect import detect
from deep_translator import GoogleTranslator

In [2]:
df = pd.read_csv('finance_data.csv')

df.head()

,title,summary,content,links,url
0,Finance,Financerefers to monetary resources and to the...,Asset (economics)\nBond\nAsset growth\nCapital...,"['/wiki/Athens', '/wiki/Quantitative_analysis_...",https://en.wikipedia.org/wiki/Finance
1,Minister of Finance (India),\nTheminister of finance(Vitta Mantrī) (or sim...,\nTheminister of finance(Vitta Mantrī) (or sim...,"['/wiki/IG_Patel', '/wiki/First_Manmohan_Singh...",https://en.wikipedia.org/wiki/Minister_of_Fina...
2,Trader (finance),"Atraderis a person, firm, or entity infinancew...",Public market\nExchange·Securities\nBond valua...,"['/wiki/Rogue_trader', '/wiki/Special-purpose_...",https://en.wikipedia.org/wiki/Trader_(finance)
3,Yahoo Finance,Yahoo Financeis a media property that is part ...,Yahoo Financeis a media property that is part ...,"['/wiki/CompuServe', '/wiki/Press_release', '/...",https://en.wikipedia.org/wiki/Yahoo_Finance
4,Equity (finance),"In finance,equityis an ownership interest inpr...",Constant purchasing power\nHistorical cost\nMa...,"['/wiki/Financial_risk', '/wiki/Financial_econ...",https://en.wikipedia.org/wiki/Equity_(finance)


In [3]:
# to check for missing values in each row on the columns
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")

title
False    9761
Name: count, dtype: int64

summary
False    8893
True      868
Name: count, dtype: int64

content
False    8897
True      864
Name: count, dtype: int64

links
False    9761
Name: count, dtype: int64

url
False    9761
Name: count, dtype: int64



In [4]:
# percentage of the missing values as per columns
summary_percentage = missing_values['summary'].value_counts() / df['summary'].size 
content_percentage = missing_values['content'].value_counts() / df['summary'].size 

print(content_percentage)
print(summary_percentage)

# 23% of our data is empty 

content
False    0.911484
True     0.088516
Name: count, dtype: float64
summary
False    0.911075
True     0.088925
Name: count, dtype: float64


In [5]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)


number of duplicate rows:  (1084, 5)


In [6]:
#removing the duplicates
df = df.drop_duplicates()

In [7]:
# check if the duplicates are gone
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
9754    False
9755    False
9757    False
9759    False
9760    False
Length: 8677, dtype: bool


In [8]:
# Drop rows where both columns 'content' and 'summary' are empty
df1=df
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')

In [9]:
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7911
Name: count, dtype: int64

summary
False    7907
True        4
Name: count, dtype: int64

content
False    7911
Name: count, dtype: int64

links
False    7911
Name: count, dtype: int64

url
False    7911
Name: count, dtype: int64



In [10]:
# after removing the duplicates 
# we see most of the raws missed data both in content and summary sections
# this being only 12 rows we can replace them with Not Available

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

# Update only rows where 'summary' is NaN
df_cleaned.loc[df['summary'].isna(), 'summary'] = df_cleaned['content'].apply(get_summary)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8136\3408418457.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned.loc[df['summary'].isna(), 'summary'] = df_cleaned['content'].apply(get_summary)


In [11]:
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")
    #it worked no more empty values

title
False    7911
Name: count, dtype: int64

summary
False    7911
Name: count, dtype: int64

content
False    7911
Name: count, dtype: int64

links
False    7911
Name: count, dtype: int64

url
False    7911
Name: count, dtype: int64



In [12]:
# as seen from the dataset above, we have to remove the \n from the summary and content column
# there are many \n in the content and the summary column
#this removes all the \n from the content table and summart
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

C:\Users\Administrator\AppData\Local\Temp\ipykernel_8136\1772156908.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_8136\1772156908.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)


In [13]:
df_cleaned = df_cleaned.drop(columns=['links'])


In [14]:
import wordninja

# Apply word splitting
df_cleaned['summary'] =df_cleaned['summary'].apply(lambda x: " ".join(wordninja.split(x)))
df_cleaned['content'] =df_cleaned['content'].apply(lambda x: " ".join(wordninja.split(x)))

In [19]:
from langdetect import detect
from deep_translator import GoogleTranslator


# Function to detect language
def detect_language(text):
    try:
        return detect(text)
    except:
        return "unknown"  # Handle errors

# Detect language
df_cleaned['Language'] = df_cleaned['content'].apply(detect_language)

# Filter only non-English rows
non_english_df = df_cleaned[df_cleaned['Language'] != 'en'].copy()  # Copy to avoid warnings

# Function to translate text
def translate_text(text):
    return GoogleTranslator(source='auto', target='en').translate(text)

# Apply translation **only to non-English rows**
non_english_df['EnglishText'] = non_english_df['Language'].apply(translate_text)

# Merge back translated texts into original DataFrame
df.update(non_english_df)


In [16]:
# removing links from the summary and content columns
#remove urls
import re

def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))


In [17]:
#resetting the index
df_cleaned = df_cleaned.reset_index(drop=True)  # drop=True removes old index

In [18]:
df_cleaned.to_csv('Finance_cleaned.csv')